In [1]:
import numpy as np
import os
import pandas as pd
import scanpy as sc

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

In [2]:
data_key_list = ["MERSCOPE_9m_AD_R1", "MERSCOPE_9m_AD_R2", "MERSCOPE_9m_WT_R1", "MERSCOPE_9m_WT_R2"]

In [3]:
# Basic information

num_cells_raw, num_cells, num_transcripts = [], [], []

for data_key in data_key_list:
    cells_raw = pd.read_csv(f"../../data/{data_key}/raw_data/metadata.csv")
    cells = pd.read_parquet(f"../../data/{data_key}/intermediate_data/cells.parquet")
    transcripts = pd.read_parquet(f"../../data/{data_key}/intermediate_data/transcripts_tmp.parquet")
    num_cells_raw.append(cells_raw.shape[0])
    num_cells.append(cells.shape[0])
    num_transcripts.append(transcripts.shape[0])

df = pd.DataFrame({"sample": data_key_list, "num_cells_raw": num_cells_raw, "num_cells": num_cells, "num_transcripts": num_transcripts})
df["transcripts_per_cell_raw"] = (df["num_transcripts"] / df["num_cells_raw"]).round(2)
df["transcripts_per_cell"] = (df["num_transcripts"] / df["num_cells"]).round(2)
df

,sample,num_cells_raw,num_cells,num_transcripts,transcripts_per_cell_raw,transcripts_per_cell
0,MERSCOPE_9m_AD_R1,453918,213792,130346112,287.16,609.69
1,MERSCOPE_9m_AD_R2,268186,177829,169508091,632.05,953.21
2,MERSCOPE_9m_WT_R1,166902,114953,147325430,882.71,1281.61
3,MERSCOPE_9m_WT_R2,168276,116985,124310192,738.73,1062.62


In [4]:
sc.pl.scatter(spots, x = "global_x_new", y = "global_y_new", color = "has_transcripts")

NameError: name 'spots' is not defined

In [5]:
# Capture efficiency dataframe

spots_dict = {}
condition, count, nc_count = [], [], []

for data_key in data_key_list:
    
    spots = sc.read_h5ad(f"../../data/{data_key}/intermediate_data/spots_raw.h5ad")
    spots.obs["has_transcripts"] = spots.X.sum(axis = 1) > 20
    spots = spots[spots.obs["has_transcripts"] == 1].copy()
    # spots = spots[spots.obs["brain_area"] != "Unknown"].copy()
    spots_dict[data_key] = spots
    nc_genes = pd.read_csv(f"../../data/{data_key}/processed_data/negative_controls.csv")
    spots_nc = spots[:, list(nc_genes["Gene"])].copy()
    
    condition += [data_key.split("MERSCOPE_")[1]] * spots.shape[0]
    count += spots.X.sum(axis = 1).flatten().tolist()
    nc_count += spots_nc.X.sum(axis = 1).flatten().tolist()

df = pd.DataFrame({"condition": condition, "count": count, "nc_count": nc_count})
df.to_csv("count_per_spot.csv", index = 0)

In [6]:
df_summary = df.groupby(["condition"]).mean()
print(df_summary["count"] / df_summary["count"].max())
print(df_summary["nc_count"] / df_summary["nc_count"].max())

condition
9m_AD_R1    0.742911
9m_AD_R2    1.000000
9m_WT_R1    0.680109
9m_WT_R2    0.886744
Name: count, dtype: float64
condition
9m_AD_R1    0.770054
9m_AD_R2    1.000000
9m_WT_R1    0.666666
9m_WT_R2    0.941667
Name: nc_count, dtype: float64


In [7]:
# Capture efficiency per gene

condition, gene_list, num_per_grid = [], [], []

for data_key in data_key_list:
    spots = sc.read_h5ad(f"../../data/{data_key}/intermediate_data/spots_raw.h5ad")
    spots.obs["has_transcripts"] = spots.X.sum(axis = 1) > 20
    spots = spots[spots.obs["has_transcripts"] == 1].copy()
    # spots = spots[spots.obs["brain_area"] != "Unknown"].copy()
    genes = spots.var["genes"].sort_values().to_list()
    transcripts = pd.read_parquet(f"../../data/{data_key}/processed_data/transcripts.parquet")
    transcripts = transcripts[transcripts["target"].isin(genes)].copy()
    condition += [data_key.split("MERSCOPE_")[1]] * len(genes)
    gene_list += genes
    num_per_grid += (transcripts.groupby("target").size().sort_index() / spots.shape[0]).to_list()

pd.DataFrame({"condition": condition, "genes": gene_list, "num_per_grid": num_per_grid}).to_csv("gene_capture_efficiency.csv", index = 0)

In [8]:
# Batch effect
pd.DataFrame({"Gene": spots_dict[data_key_list[0]].var.index, data_key_list[0]: np.mean(spots_dict[data_key_list[0]].X, axis = 0), data_key_list[1]: np.mean(spots_dict[data_key_list[1]].X, axis = 0)}).to_csv(f"count_per_gene_{data_key_list[0].split('_R')[0]}.csv", index = 0)
pd.DataFrame({"Gene": spots_dict[data_key_list[2]].var.index, data_key_list[2]: np.mean(spots_dict[data_key_list[2]].X, axis = 0), data_key_list[3]: np.mean(spots_dict[data_key_list[3]].X, axis = 0)}).to_csv(f"count_per_gene_{data_key_list[2].split('_R')[0]}.csv", index = 0)